# Valor crítico do teste de cointegração não paramétrica — por simulação

**Para que serve.** A formação própria do método NP (`pairs_formation_np_propria.ipynb`) aprova
um par se a estatística ADF dos **erros fora da amostra** da curva for "negativa o bastante". Este
notebook descobre quanto é "o bastante": o valor que a estatística atinge **por acaso** em pares
**sem** cointegração.

### Por que não usar uma tabela

| ADF aplicado a… | valor crítico 5% | referência |
|---|---|---|
| uma série de preços | ≈ −2,86 | Dickey & Fuller (1979); MacKinnon (1996) |
| resíduo de uma **reta** estimada por OLS | ≈ −3,34 (2 variáveis) | Engle & Granger (1987); MacKinnon (1991, 2010) |
| erro de uma **curva** kernel com h escolhido por CV | **não há tabela** | — |

A curva se adapta aos dados muito mais que uma reta; com h pequeno ela "cola" no caminho, e o erro
parece estacionário mesmo sem equilíbrio — a regressão não paramétrica espúria de Phillips (2009).
Quanto isso distorce depende do kernel, de como o h é escolhido, do tamanho da amostra e de o erro
ser dentro ou fora da amostra. Por isso os slides da disciplina (21–22) tratam ADF/PP/KPSS como
diagnóstico complementar, e Wang & Phillips (2012) — o teste formal mais próximo — testam a **forma**
da relação supondo que ela existe, não a existência.

### Como resolvemos: a mesma receita das tabelas, com o nosso procedimento

As tabelas de MacKinnon (1991, 2010) são resultado de simulação: gera-se pares **sem** cointegração,
roda-se o procedimento, registra-se a estatística e lê-se o quantil de 5%. Fazemos o mesmo, com o
**nosso** procedimento no lugar do OLS: `kernel_np.teste_coint_np` — a mesma função que a formação
própria usa, o que é condição para o valor crítico valer.

### A hipótese nula, a partir dos nossos dados

**Nula principal — pares reais com os dias embaralhados.** Sorteamos pares reais entre os candidatos
do pré-filtro SSD (os mesmos que chegam ao teste na formação), pegamos os retornos diários das duas
ações e **embaralhamos a ordem dos dias, mantendo juntos os dois retornos de cada dia**:

- fica: a correlação do mesmo dia, as caudas e a volatilidade reais;
- some: o mecanismo de volta ao equilíbrio, que está na **sequência** dos dias (o spread se afasta e
  volta nos dias seguintes). Os preços reconstruídos andam juntos, mas não têm equilíbrio — é a nula.

**Nula de conferência — passeios aleatórios gaussianos** com a mesma volatilidade e correlação de cada
par sorteado: o caso idealizado das tabelas. Se as duas nulas concordarem, o valor crítico é robusto.

Limitação declarada: embaralhar os dias também apaga o agrupamento de volatilidade (*volatility
clustering*). E o teste é aplicado a centenas de pares por janela e os 20 melhores são escolhidos —
o problema de testes múltiplos que a formação linear também tem; o valor crítico controla o erro por
par, não o da seleção.

### Poder

Um teste que nunca aprova nada é inútil. Na seção 5, simulamos pares que **são** cointegrados ao
longo de uma curva e medimos com que frequência o teste os detecta.

In [ ]:
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kernel_np import teste_coint_np
from formacao_utils import filtrar_ativos_completos, normalizar_precos, candidatos_ssd, janelas_formacao

warnings.filterwarnings('ignore')

# ── Procedimento testado — TEM de ser igual ao da formação própria ───────────
# (a formação confere estes valores contra o JSON salvo aqui e recusa rodar se divergirem)
GRADE_H_REL      = np.r_[np.geomspace(0.05, 5, 25), np.inf]   # = pairs_formation_np.ipynb
GAP_UNILATERAL   = 20      # CV unilateral (DECISOES.md § 6.1)
T_MIN_UNILATERAL = 126
AUTOLAG_ADF      = 'AIC'   # = formação linear

# ── Amostra da hipótese nula ──────────────────────────────────────────────────
N_PRE_SSD     = 300   # candidatos de onde os pares reais são sorteados (= formação própria)
PASSO_JANELAS = 5     # usa 1 a cada 5 janelas de formação mensais → ~80 janelas de 1991 a 2025
N_REAL        = 3000  # pares da nula principal (reais com dias embaralhados)
N_GAUSS       = 2000  # pares da nula de conferência (gaussiana)
N_PODER       = 200   # pares cointegrados simulados por cenário de poder
SEMENTE       = 20261024
NIVEIS        = [0.01, 0.05, 0.10]
CRIT_EG_5     = -3.34  # Engle–Granger / MacKinnon, 2 variáveis, 5% (o da formação linear)
CRIT_DF_5     = -2.86  # Dickey–Fuller com constante, 5%

SAIDA = 'output/valor_critico'
META_COLS = ['year', 'semester', 'period', 'day_in_semester', 'total_days_sem']

import os
os.makedirs(SAIDA, exist_ok=True)
rng = np.random.default_rng(SEMENTE)

## 1. Dados e janelas

In [ ]:
df = pd.read_csv('../../data_bases/base_unificada.csv', parse_dates=['date']).set_index('date')
colunas = [c for c in df.columns if c not in META_COLS]
meses_linha = pd.PeriodIndex(df.index, freq='M')

janelas = janelas_formacao(df.index, 12, 6, 1)
jan_sel = janelas[::PASSO_JANELAS]
print(f"Janelas de formação: {len(janelas)} | usadas na simulação: {len(jan_sel)} "
      f"({jan_sel[0][0][0]} → {jan_sel[-1][0][-1]})")

## 2. Sorteio dos pares reais

Em cada janela usada, os pares são sorteados **entre os `N_PRE_SSD` candidatos de menor SSD** — os
mesmos que a formação própria testa. Assim a nula tem o perfil dos pares que de fato chegam ao teste
(ações que andam juntas), e não de pares quaisquer do índice.

In [ ]:
por_janela = int(np.ceil(N_REAL / len(jan_sel)))
amostra = []
t0 = time.time()
for mf, _ in jan_sel:
    precos = filtrar_ativos_completos(df.loc[meses_linha.isin(mf), colunas])
    Q = normalizar_precos(precos)
    cands = candidatos_ssd(Q, N_PRE_SSD)
    for k in rng.choice(len(cands), size=min(por_janela, len(cands)), replace=False):
        a, b, ssd = cands[k]
        r = precos[[a, b]].pct_change().dropna().values
        amostra.append({'janela': f"{mf[0]}_a_{mf[-1]}", 'ativo1': a, 'ativo2': b,
                        'rank_ssd': int(k), 'r': r, 'corr': float(np.corrcoef(r.T)[0, 1])})
amostra = amostra[:N_REAL]
print(f"Pares sorteados: {len(amostra):,} em {time.time() - t0:.0f} s")
print(f"Correlação diária dos pares: mediana {np.median([s['corr'] for s in amostra]):.2f} "
      f"(p10 {np.percentile([s['corr'] for s in amostra], 10):.2f}, p90 {np.percentile([s['corr'] for s in amostra], 90):.2f})")

## 3. Simulação sob a hipótese nula

Para cada par: reconstrói os preços a partir dos retornos (embaralhados, ou gaussianos), começando em
1, e roda `teste_coint_np` exatamente como na formação — as duas direções, h pela CV unilateral,
ADF dos erros um passo à frente.

In [ ]:
def caminho(r):
    """Retornos diários (n × 2) → preços normalizados começando em 1 ((n+1) × 2)."""
    return np.vstack([np.ones((1, 2)), np.cumprod(1 + r, axis=0)])


def roda(Q):
    return teste_coint_np(Q[:, 0], Q[:, 1], GRADE_H_REL, GAP_UNILATERAL, T_MIN_UNILATERAL, AUTOLAG_ADF)


linhas = []
t0 = time.time()
for k, s in enumerate(amostra):
    # nula principal: mesmos retornos, dias embaralhados (os dois retornos de cada dia ficam juntos)
    r_emb = s['r'][rng.permutation(len(s['r']))]
    res = roda(caminho(r_emb))
    linhas.append({'nula': 'real_embaralhada', **{c: s[c] for c in ('janela', 'ativo1', 'ativo2', 'rank_ssd', 'corr')},
                   'stat': res['stat'], 'h_linear': res['h_linear'], 'h_rel': res['h_rel']})
    # nula de conferência: gaussiana com a média, a volatilidade e a correlação do par
    if k < N_GAUSS:
        r_g = rng.multivariate_normal(s['r'].mean(axis=0), np.cov(s['r'].T), size=len(s['r']))
        res = roda(caminho(r_g))
        linhas.append({'nula': 'gaussiana', **{c: s[c] for c in ('janela', 'ativo1', 'ativo2', 'rank_ssd', 'corr')},
                       'stat': res['stat'], 'h_linear': res['h_linear'], 'h_rel': res['h_rel']})
    if (k + 1) % 500 == 0:
        print(f"  {k + 1:5,} / {len(amostra):,}  ({time.time() - t0:4.0f} s)")

nula = pd.DataFrame(linhas).dropna(subset=['stat'])
nula.to_csv(f'{SAIDA}/simulacao_nula.csv', index=False)
print(f"Simulações válidas: {nula.groupby('nula').size().to_dict()}  | {time.time() - t0:.0f} s")

## 4. Valores críticos e o erro que as tabelas cometeriam

- **quantis**: o valor crítico a 1%, 5% e 10% de cada nula;
- **% abaixo de −3,34 / −2,86**: com que frequência pares **sem** cointegração passariam se usássemos
  a tabela de Engle–Granger ou a de Dickey–Fuller. Num teste a 5% calibrado, isso deveria ser 5%.

In [ ]:
def resumo(d):
    s = d['stat']
    return pd.Series({**{f'crítico {int(n * 100)}%': s.quantile(n) for n in NIVEIS},
                      'passariam com −3,34 (EG)': (s < CRIT_EG_5).mean(),
                      'passariam com −2,86 (DF)': (s < CRIT_DF_5).mean(),
                      'reta escolhida (h = ∞)': d['h_linear'].mean(),
                      'pares': len(d)})


tab = nula.groupby('nula').apply(resumo).T
display(tab.style.format('{:.3f}'))

CRIT = {f'{int(n * 100)}%': float(nula.loc[nula['nula'] == 'real_embaralhada', 'stat'].quantile(n)) for n in NIVEIS}
CRIT_G = {f'{int(n * 100)}%': float(nula.loc[nula['nula'] == 'gaussiana', 'stat'].quantile(n)) for n in NIVEIS}
print(f"\nValor crítico a 5% (nula principal): {CRIT['5%']:.3f}   | gaussiana: {CRIT_G['5%']:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bins = np.linspace(nula['stat'].quantile(0.001), nula['stat'].quantile(0.999), 60)
for nome, cor in [('real_embaralhada', '#E65100'), ('gaussiana', '#607D8B')]:
    ax.hist(nula.loc[nula['nula'] == nome, 'stat'], bins=bins, alpha=0.55, color=cor, label=nome.replace('_', ' '))
for v, lab, ls in [(CRIT['5%'], f"crítico simulado 5% ({CRIT['5%']:.2f})", '-'), (CRIT_EG_5, 'Engle–Granger 5% (−3,34)', '--'),
                   (CRIT_DF_5, 'Dickey–Fuller 5% (−2,86)', ':')]:
    ax.axvline(v, color='black', ls=ls, lw=1.3, label=lab)
ax.set_xlabel('estatística ADF dos erros fora da amostra (pares SEM cointegração)')
ax.set_ylabel('pares')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAIDA}/distribuicao_nula.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Poder: o teste detecta cointegração quando ela existe?

Pares cointegrados por construção: o regressor $x$ é o caminho (embaralhado) de uma ação real — um
passeio aleatório com cara de ação — e

$$y_t = 1 + 0{,}9\,(x_t - 1) + c\,(x_t - 1)^2 + u_t, \qquad u_t = \varphi\, u_{t-1} + \varepsilon_t$$

- $c = 0$: equilíbrio **linear**; $c = -1$: equilíbrio **curvo** (côncavo);
- $\varphi \in \{0{,}80;\ 0{,}90;\ 0{,}95\}$: meias-vidas de ~3, ~7 e ~14 pregões;
- desvio do erro de equilíbrio $u$ = 0,02, a ordem de grandeza do σ do spread nos pares reais.

Mede-se a fração detectada pelo valor crítico simulado (e, para referência, por −3,34).

In [ ]:
SIGMA_U = 0.02
linhas_p = []
t0 = time.time()
for phi in (0.80, 0.90, 0.95):
    for c in (0.0, -1.0):
        for k in range(N_PODER):
            s = amostra[rng.integers(len(amostra))]
            x = caminho(s['r'][rng.permutation(len(s['r']))])[:, 0]
            u = np.empty(len(x))
            u[0] = rng.normal(0, SIGMA_U)
            eps = rng.normal(0, SIGMA_U * np.sqrt(1 - phi ** 2), len(x))
            for t in range(1, len(x)):
                u[t] = phi * u[t - 1] + eps[t]
            y = 1 + 0.9 * (x - 1) + c * (x - 1) ** 2 + u
            res = teste_coint_np(x, y, GRADE_H_REL, GAP_UNILATERAL, T_MIN_UNILATERAL, AUTOLAG_ADF)
            linhas_p.append({'phi': phi, 'meia_vida': np.log(0.5) / np.log(phi), 'curva': 'curva' if c else 'reta',
                             'stat': res['stat'], 'h_linear': res['h_linear']})
poder = pd.DataFrame(linhas_p).dropna(subset=['stat'])
poder.to_csv(f'{SAIDA}/simulacao_poder.csv', index=False)
tab_poder = poder.groupby(['curva', 'phi']).agg(
    meia_vida=('meia_vida', 'first'),
    detectados_simulado=('stat', lambda s: (s < CRIT['5%']).mean()),
    detectados_EG=('stat', lambda s: (s < CRIT_EG_5).mean()),
    reta_escolhida=('h_linear', 'mean'))
display(tab_poder.style.format({'meia_vida': '{:.1f}', 'detectados_simulado': '{:.1%}', 'detectados_EG': '{:.1%}', 'reta_escolhida': '{:.1%}'}))
print(f"{time.time() - t0:.0f} s")

## 6. Salvando o valor crítico

In [ ]:
saida = {
    'valor_critico': CRIT,
    'valor_critico_gaussiana': CRIT_G,
    'nula_principal': 'real_embaralhada',
    'n_simulacoes': nula.groupby('nula').size().to_dict(),
    'falso_positivo_com_EG_334': float((nula.loc[nula['nula'] == 'real_embaralhada', 'stat'] < CRIT_EG_5).mean()),
    'parametros': {'grade_h_rel': [None if np.isinf(v) else float(v) for v in GRADE_H_REL],
                   'gap_unilateral': GAP_UNILATERAL, 't_min_unilateral': T_MIN_UNILATERAL,
                   'autolag_adf': AUTOLAG_ADF, 'n_pre_ssd': N_PRE_SSD, 'semente': SEMENTE},
    'gerado_em': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
}
with open(f'{SAIDA}/valor_critico_np.json', 'w', encoding='utf-8') as f:
    json.dump(saida, f, ensure_ascii=False, indent=1)
print(json.dumps({k: saida[k] for k in ('valor_critico', 'valor_critico_gaussiana', 'falso_positivo_com_EG_334')}, indent=1))

### Referências

- Dickey, D. & Fuller, W. (1979). *Distribution of the estimators for autoregressive time series with a unit root.* JASA 74.
- Engle, R. & Granger, C. (1987). *Co-integration and error correction.* Econometrica 55(2) — teste em dois passos (slides 6–7).
- MacKinnon, J. (1991, 2010). *Critical values for cointegration tests* — os valores críticos de Engle–Granger são obtidos por simulação; mesmo método usado aqui.
- Phillips, P. (2009). *Local limit theory and spurious nonparametric regression.* Econometric Theory 25(6) — por que a curva kernel "inventa" cointegração.
- Wang, Q. & Phillips, P. (2012). *A specification test for nonlinear nonstationary models.* Annals of Statistics 40(2) — teste de **forma** (linear × curva), não de existência; usado como diagnóstico, não como seleção.
- Slides *Cointegração Não Paramétrica via Kernel* (ECO 02091, UFRGS), 21–22: ADF/PP/KPSS são diagnóstico complementar sob regressão não paramétrica.